# 06a_baseline_beta_lexicon_auto

## Objetivo
Crear un baseline beta de etiquetado automático usando coincidencias directas contra
`lexicons/processed/hatecr_lexicon.csv`.

Este notebook no reemplaza el etiquetado manual ni el modelo supervisado. Produce una
etiqueta débil (`auto_lexicon_hostility_label`) para diagnóstico y la compara con
`y_hostility`, que conserva la decisión humana binaria `manual_hostility`, cuando las
tres etiquetas manuales están completas y son coherentes.


## Advertencia metodológica
Una coincidencia léxica no equivale a hostilidad ni a discurso de odio. Este baseline es deliberadamente transparente y útil como punto de partida, pero requiere revisión manual por falsos positivos, ironía, citas, negaciones y contexto político.


In [ ]:
import importlib
import os
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_STATE = 42
AUTO_LEXICON_MIN_TERM_CHARS = int(os.getenv("AUTO_LEXICON_MIN_TERM_CHARS", "4"))
AUTO_LEXICON_MIN_HITS = int(os.getenv("AUTO_LEXICON_MIN_HITS", "1"))
AUTO_LEXICON_MIN_SCORE = float(os.getenv("AUTO_LEXICON_MIN_SCORE", "1.0"))
AUTO_LEXICON_EXCLUDE_UNCATEGORIZED = os.getenv(
    "AUTO_LEXICON_EXCLUDE_UNCATEGORIZED", "true"
).strip().lower() == "true"
AUTO_LEXICON_MATCH_LEMMA = os.getenv("AUTO_LEXICON_MATCH_LEMMA", "true").strip().lower() == "true"

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True


In [ ]:
def is_project_dir(path):
    return (path / "config").exists() and (path / "data").exists() and (path / "lexicons").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_dir(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_dir(child):
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.labels as label_utils
importlib.reload(label_utils)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
LEXICON_PROCESSED = PROJECT_ROOT / "lexicons" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FORMAL_EDA_REPORTS = REPORTS_DIR / "formal_eda"
FIGURES_DIR = REPORTS_DIR / "figures"
MANUAL_SAMPLE_PATH = FORMAL_EDA_REPORTS / "manual_review_sample.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("LEXICON_PROCESSED:", LEXICON_PROCESSED)
print("MANUAL_SAMPLE_PATH (solo lectura):", MANUAL_SAMPLE_PATH)


## 1. Carga del corpus y del lexicón


In [ ]:
def safe_read_csv(path, name, **kwargs):
    if not path.exists():
        print(f"[WARN] Falta {name}: {path}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, **kwargs)
        for id_column in ["tweet_id", "reply_id", "source_post_id"]:
            if id_column in df.columns:
                df[id_column] = df[id_column].astype("string")
        print(f"[OK] {name}: {len(df):,} filas, {len(df.columns):,} columnas")
        return df
    except Exception as exc:
        print(f"[WARN] Error leyendo {name}: {exc}")
        return pd.DataFrame()


def ensure_col(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    return df


corpus_candidates = [
    DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_eda.csv",
    DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_analysis.csv",
    DATA_PROCESSED / "x_media_anchored_interactions_corpus_eda_dedup.csv",
    DATA_PROCESSED / "x_media_anchored_interactions_corpus_eda.csv",
]

corpus_df = pd.DataFrame()
corpus_path = None
for candidate_path in corpus_candidates:
    candidate_df = safe_read_csv(candidate_path, candidate_path.name)
    if not candidate_df.empty:
        corpus_df = candidate_df
        corpus_path = candidate_path
        break

if corpus_df.empty:
    raise FileNotFoundError("No se encontró corpus para etiquetado automático.")

lexicon_path = LEXICON_PROCESSED / "hatecr_lexicon.csv"
lexicon_df = safe_read_csv(lexicon_path, "hatecr_lexicon")
if lexicon_df.empty:
    raise FileNotFoundError(f"No se encontró lexicón procesado: {lexicon_path}")

manual_sample_df = safe_read_csv(
    MANUAL_SAMPLE_PATH,
    "muestra manual formal (solo lectura)",
)

print("Corpus usado:", corpus_path)


## 2. Normalización de texto y filtrado del lexicón


In [ ]:
def normalize_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    value = str(text).lower().strip()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"#(\w+)", r"\1", value)
    value = unicodedata.normalize("NFD", value)
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = re.sub(r"[^a-z0-9ñü\s]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def to_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    return text in {"1", "true", "t", "yes", "y", "si", "sí"}


def parse_severity(value):
    if pd.isna(value):
        return 1.0
    try:
        return float(value)
    except Exception:
        mapping = {
            "low": 1.0,
            "baja": 1.0,
            "medium": 2.0,
            "media": 2.0,
            "high": 3.0,
            "alta": 3.0,
            "very_high": 4.0,
            "muy_alta": 4.0,
        }
        return float(mapping.get(str(value).strip().lower(), 1.0))


for col in ["text", "text_norm", "tweet_id", "source_type", "anchor_media_handle", "event_id"]:
    corpus_df = ensure_col(corpus_df, col)

base_text = corpus_df["text_norm"].where(corpus_df["text_norm"].notna(), corpus_df["text"])
corpus_df["text_model_norm"] = base_text.fillna("").astype(str).map(normalize_text)

lexicon_df.columns = [c.strip().lower() for c in lexicon_df.columns]
for col in ["term", "lemma", "category", "severity", "include_in_classification"]:
    lexicon_df = ensure_col(lexicon_df, col)

lex_work = lexicon_df.copy()
lex_work["include_in_classification_bool"] = lex_work["include_in_classification"].map(to_bool)
if lex_work["include_in_classification_bool"].any():
    lex_work = lex_work[lex_work["include_in_classification_bool"]].copy()

if AUTO_LEXICON_EXCLUDE_UNCATEGORIZED:
    lex_work = lex_work[lex_work["category"].fillna("").astype(str).str.lower() != "uncategorized"].copy()

term_sources = ["term"]
if AUTO_LEXICON_MATCH_LEMMA:
    term_sources.append("lemma")

term_rows = []
for _, row in lex_work.iterrows():
    for source_col in term_sources:
        raw_term = row.get(source_col)
        term_norm = normalize_text(raw_term)
        if not term_norm:
            continue
        if len(term_norm.replace(" ", "")) < AUTO_LEXICON_MIN_TERM_CHARS:
            continue
        if term_norm.isdigit():
            continue
        term_rows.append(
            {
                "term_norm": term_norm,
                "category": str(row.get("category") or "unknown").strip().lower(),
                "severity": parse_severity(row.get("severity")),
                "source_field": source_col,
            }
        )

terms_df = pd.DataFrame(term_rows).drop_duplicates(subset=["term_norm", "category"]).reset_index(drop=True)
print("Términos activos para baseline beta:", len(terms_df))
print("Categorías activas:", sorted(terms_df["category"].dropna().unique().tolist())[:40])
display(terms_df.head(10))


## 3. Etiquetado automático por coincidencia lexicográfica


In [ ]:
term_to_categories = defaultdict(set)
term_to_severity = {}
for _, row in terms_df.iterrows():
    term = row["term_norm"]
    term_to_categories[term].add(row["category"])
    term_to_severity[term] = max(float(row["severity"]), float(term_to_severity.get(term, 0.0)))

single_terms = {t for t in term_to_categories if " " not in t}
multi_terms = {t for t in term_to_categories if " " in t}


def match_lexicon(text_norm):
    text = str(text_norm or "")
    if not text:
        return {
            "terms": [],
            "categories": [],
            "score": 0.0,
        }

    tokens = set(text.split())
    matched_terms = set()

    for term in single_terms:
        if term in tokens:
            matched_terms.add(term)

    padded = f" {text} "
    for term in multi_terms:
        if f" {term} " in padded:
            matched_terms.add(term)

    categories = set()
    score = 0.0
    for term in matched_terms:
        categories.update(term_to_categories.get(term, set()))
        score += float(term_to_severity.get(term, 1.0))

    return {
        "terms": sorted(matched_terms),
        "categories": sorted(categories),
        "score": round(score, 3),
    }


matches = corpus_df["text_model_norm"].map(match_lexicon)
corpus_df["auto_lexicon_hits"] = matches.map(lambda x: len(x["terms"]))
corpus_df["auto_lexicon_terms_found"] = matches.map(lambda x: "|".join(x["terms"]))
corpus_df["auto_lexicon_categories_found"] = matches.map(lambda x: "|".join(x["categories"]))
corpus_df["auto_lexicon_score"] = matches.map(lambda x: x["score"])
corpus_df["auto_lexicon_hostility_label"] = (
    (corpus_df["auto_lexicon_hits"] >= AUTO_LEXICON_MIN_HITS)
    & (corpus_df["auto_lexicon_score"] >= AUTO_LEXICON_MIN_SCORE)
).astype(int)
corpus_df["auto_lexicon_label_rule"] = (
    f"hits>={AUTO_LEXICON_MIN_HITS};score>={AUTO_LEXICON_MIN_SCORE};"
    f"min_chars={AUTO_LEXICON_MIN_TERM_CHARS};exclude_uncategorized={AUTO_LEXICON_EXCLUDE_UNCATEGORIZED}"
)

summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(corpus_df)},
        {"metric": "auto_positive", "value": int(corpus_df["auto_lexicon_hostility_label"].sum())},
        {"metric": "auto_negative", "value": int((corpus_df["auto_lexicon_hostility_label"] == 0).sum())},
        {"metric": "auto_positive_pct", "value": round(corpus_df["auto_lexicon_hostility_label"].mean() * 100, 2)},
        {"metric": "active_terms", "value": len(terms_df)},
    ]
)
display(summary)


## 4. Comparación con etiquetas manuales si existen

La comparación usa `y_hostility`, copia normalizada de `manual_hostility`. El nivel
`0–3` y `manual_hate_speech` se mantienen como variables humanas separadas. El archivo
canónico se lee en modo de solo lectura y no se modifica.


In [ ]:
def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    accuracy = (tp + tn) / len(y_true) if len(y_true) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "n_eval": int(len(y_true)),
        "true_positive": tp,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "accuracy": round(accuracy, 4),
        "precision_hostile": round(precision, 4),
        "recall_hostile": round(recall, 4),
        "f1_hostile": round(f1, 4),
    }


if manual_sample_df.empty:
    eval_df = pd.DataFrame()
    annotation_diagnostics = None
else:
    prepared_manual_df, annotation_diagnostics = label_utils.prepare_manual_annotations(
        manual_sample_df,
        strict=False,
    )
    manual_ready_df = prepared_manual_df[
        prepared_manual_df["annotation_ready_for_training"]
    ][[
        "tweet_id", "hostility_relevance_normalized", "hostility_level_label",
        "y_hostility", "y_severe_or_identity", "y_hate_speech",
    ]].copy()
    manual_ready_df = manual_ready_df.rename(columns={"y_hostility": "y_hostility_manual"})
    eval_df = corpus_df.merge(manual_ready_df, on="tweet_id", how="inner")

if eval_df.empty:
    metrics_df = pd.DataFrame()
    confusion_df = pd.DataFrame()
    false_positives = pd.DataFrame()
    false_negatives = pd.DataFrame()
    print("No hay niveles manuales válidos para comparar.")
else:
    metrics_df = pd.DataFrame([
        binary_metrics(
            eval_df["y_hostility_manual"].astype(int),
            eval_df["auto_lexicon_hostility_label"].astype(int),
        )
    ])
    metrics_df.insert(0, "model", "beta_auto_lexicon")
    metrics_df.insert(1, "manual_target", "y_hostility_from_level_0_3")

    confusion_df = pd.DataFrame(
        [
            {"manual": 0, "auto": 0, "n": int(((eval_df["y_hostility_manual"] == 0) & (eval_df["auto_lexicon_hostility_label"] == 0)).sum())},
            {"manual": 0, "auto": 1, "n": int(((eval_df["y_hostility_manual"] == 0) & (eval_df["auto_lexicon_hostility_label"] == 1)).sum())},
            {"manual": 1, "auto": 0, "n": int(((eval_df["y_hostility_manual"] == 1) & (eval_df["auto_lexicon_hostility_label"] == 0)).sum())},
            {"manual": 1, "auto": 1, "n": int(((eval_df["y_hostility_manual"] == 1) & (eval_df["auto_lexicon_hostility_label"] == 1)).sum())},
        ]
    )

    false_positives = eval_df[(eval_df["y_hostility_manual"] == 0) & (eval_df["auto_lexicon_hostility_label"] == 1)].copy()
    false_negatives = eval_df[(eval_df["y_hostility_manual"] == 1) & (eval_df["auto_lexicon_hostility_label"] == 0)].copy()

    display(metrics_df)
    display(confusion_df)
    print("Falsos positivos:", len(false_positives))
    print("Falsos negativos:", len(false_negatives))


## 5. Resúmenes por término, categoría, medio y tipo de fuente


In [ ]:
term_counter = Counter()
category_counter = Counter()

for terms in corpus_df["auto_lexicon_terms_found"].fillna("").astype(str):
    for term in terms.split("|"):
        if term:
            term_counter[term] += 1

for cats in corpus_df["auto_lexicon_categories_found"].fillna("").astype(str):
    for cat in cats.split("|"):
        if cat:
            category_counter[cat] += 1

matched_terms_df = pd.DataFrame(term_counter.most_common(), columns=["term", "n_texts"])
category_summary_df = pd.DataFrame(category_counter.most_common(), columns=["category", "n_texts"])

by_source_type = (
    corpus_df.groupby("source_type", dropna=False)["auto_lexicon_hostility_label"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"count": "n_rows", "sum": "auto_positive", "mean": "auto_positive_rate"})
)
by_source_type["auto_positive_rate"] = by_source_type["auto_positive_rate"].round(4)

by_anchor_media = (
    corpus_df.groupby("anchor_media_handle", dropna=False)["auto_lexicon_hostility_label"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"count": "n_rows", "sum": "auto_positive", "mean": "auto_positive_rate"})
    .sort_values("n_rows", ascending=False)
)
by_anchor_media["auto_positive_rate"] = by_anchor_media["auto_positive_rate"].round(4)

display(matched_terms_df.head(30))
display(category_summary_df.head(30))
display(by_source_type)
display(by_anchor_media.head(20))


## 6. Gráficos exploratorios


In [ ]:
fig_paths = {}

label_counts = corpus_df["auto_lexicon_hostility_label"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(label_counts.index.astype(str), label_counts.values, color=["#5f6c7b", "#c0392b"][: len(label_counts)])
plt.title("Distribución etiqueta automática por lexicón")
plt.xlabel("auto_lexicon_hostility_label")
plt.ylabel("N textos")
plt.tight_layout()
fig_paths["label_distribution"] = FIGURES_DIR / "beta_lexicon_label_distribution.png"
plt.savefig(fig_paths["label_distribution"], dpi=150)
plt.show()

if not matched_terms_df.empty:
    top_terms = matched_terms_df.head(30).sort_values("n_texts", ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(top_terms["term"], top_terms["n_texts"], color="#8e3b46")
    plt.title("Top términos detectados por lexicón")
    plt.xlabel("N textos")
    plt.tight_layout()
    fig_paths["top_terms"] = FIGURES_DIR / "beta_lexicon_top_terms.png"
    plt.savefig(fig_paths["top_terms"], dpi=150)
    plt.show()

if not eval_df.empty:
    matrix = np.array([
        [int(((eval_df["y_hostility_manual"] == 0) & (eval_df["auto_lexicon_hostility_label"] == 0)).sum()),
         int(((eval_df["y_hostility_manual"] == 0) & (eval_df["auto_lexicon_hostility_label"] == 1)).sum())],
        [int(((eval_df["y_hostility_manual"] == 1) & (eval_df["auto_lexicon_hostility_label"] == 0)).sum()),
         int(((eval_df["y_hostility_manual"] == 1) & (eval_df["auto_lexicon_hostility_label"] == 1)).sum())],
    ])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["auto 0", "auto 1"])
    ax.set_yticklabels(["manual 0", "manual 1"])
    ax.set_xlabel("Automático")
    ax.set_ylabel("Manual derivado")
    ax.set_title("Lexicón beta vs y_hostility manual derivado")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(matrix[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig_paths["confusion_manual"] = FIGURES_DIR / "beta_lexicon_confusion_manual.png"
    plt.savefig(fig_paths["confusion_manual"], dpi=150)
    plt.show()

fig_paths


## 7. Exportación de resultados


In [ ]:
out_corpus_path = DATA_PROCESSED / "x_media_anchored_interactions_corpus_beta_lexicon_labels.csv"
summary_path = REPORTS_DIR / "beta_lexicon_auto_label_summary.csv"
terms_path = REPORTS_DIR / "beta_lexicon_matched_terms.csv"
category_path = REPORTS_DIR / "beta_lexicon_category_summary.csv"
source_type_path = REPORTS_DIR / "beta_lexicon_by_source_type.csv"
anchor_path = REPORTS_DIR / "beta_lexicon_by_anchor_media.csv"
metrics_path = REPORTS_DIR / "beta_lexicon_vs_manual_metrics.csv"
confusion_path = REPORTS_DIR / "beta_lexicon_confusion_manual.csv"
fp_path = REPORTS_DIR / "beta_lexicon_false_positives.csv"
fn_path = REPORTS_DIR / "beta_lexicon_false_negatives.csv"

corpus_df.to_csv(out_corpus_path, index=False)
summary.to_csv(summary_path, index=False)
matched_terms_df.to_csv(terms_path, index=False)
category_summary_df.to_csv(category_path, index=False)
by_source_type.to_csv(source_type_path, index=False)
by_anchor_media.to_csv(anchor_path, index=False)

if not metrics_df.empty:
    metrics_df.to_csv(metrics_path, index=False)
    confusion_df.to_csv(confusion_path, index=False)
    false_positives.to_csv(fp_path, index=False)
    false_negatives.to_csv(fn_path, index=False)

print("Archivos guardados:")
for path in [out_corpus_path, summary_path, terms_path, category_path, source_type_path, anchor_path]:
    print("-", path)
if not metrics_df.empty:
    for path in [metrics_path, confusion_path, fp_path, fn_path]:
        print("-", path)

print("\nFiguras:")
for _, path in fig_paths.items():
    print("-", path)


## 8. Lectura recomendada de resultados

Revisar especialmente precisión y recall contra `y_hostility`, que corresponde a
`manual_hostility` normalizada. Si el recall es alto pero la precisión baja, el lexicón
puede servir como detector amplio para priorizar revisión. La evaluación de discurso de
odio debe hacerse por separado contra `manual_hate_speech`; no debe inferirse a partir
de esta etiqueta automática de hostilidad.
